##Data Loading

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
cell_df = pd.read_parquet('/content/drive/MyDrive/Datasets/Berlin_v2x/cellular_dataframe.parquet')
#Preprocess the data
#Convert the datarate for MBps
cell_df['datarate']=cell_df['datarate']/1e6

## Problem setup - filters



In [ ]:
print(cell_df.shape)#Shape of the Original Dataset

#filter for downlink and uses datarate as qos measurement - the original dataset has downlink and uplink for direction, and for measured_qos has 'datarate' and 'delay'
filtered_data = cell_df.query("direction == 'downlink' & measured_qos == 'datarate'")
print(filtered_data.shape)#Show shape of the Filtered Data

# Remove incomplete measurements without datarate
filtered_data = filtered_data.dropna(subset='datarate')
print(filtered_data.shape)#Show shape of the Filtered Data

#Remove stationary measurements (speed=0)
filtered_data = filtered_data.query("speed_kmh != 0")
print(filtered_data.shape)#Show shape of the Filtered Data

#Remove incomplete measurements without speed (speed=NaN)
filtered_data = filtered_data.dropna(subset='speed_kmh')
print(filtered_data.shape)#Show shape of the Filtered Data

#Filter for A3D scenario - just to confirm, but the filters above already did that
filtered_data = filtered_data.query("scenario == 'A3D'")
print(filtered_data.shape)#Show shape of the Filtered Data


(207434, 159)
(65802, 159)
(52673, 159)
(32180, 159)
(32060, 159)
(32060, 159)


### Create datasets for each environment (area)

In [ ]:
data_avenue= filtered_data.query("area == 'Avenue'")
print(data_avenue.shape)
data_park=filtered_data.query("area == 'Park'")
print(data_park.shape)
data_residential=filtered_data.query("area == 'Residential'")
print(data_residential.shape)
data_highway=filtered_data.query("area == 'Highway'")
print(data_highway.shape)
data_tunnel=filtered_data.query("area == 'Tunnel'")
print(data_tunnel.shape)



(7066, 159)
(11698, 159)
(8580, 159)
(4527, 159)
(189, 159)


## Feature selection and Split

In [ ]:
qos_column = 'datarate'

feature_columns = [
    #PHY features
     'PCell_RSRP_max',
     'PCell_RSRQ_max',
     'PCell_RSSI_max',
     'PCell_SNR_1',
     'PCell_SNR_2',
     'SCell_RSRP_max',
     'SCell_RSRQ_max',
     'SCell_RSSI_max',
     'SCell_SNR_1',
     'SCell_SNR_2',

    #Physical Shared Channels and Radio Resource Control (RRC)
     'PCell_Downlink_Num_RBs',
     'PCell_Downlink_TB_Size',
     'PCell_Downlink_Average_MCS',
     'PCell_Downlink_bandwidth_MHz',
     'PCell_Cell_Identity',
     'PCell_freq_MHz',
     'SCell_Downlink_Num_RBs',
     'SCell_Downlink_TB_Size',
     'SCell_Downlink_Average_MCS',
     'SCell_Downlink_bandwidth_MHz',
     'SCell_Cell_Identity',
     'SCell_freq_MHz'
     ]

used_features = len(feature_columns)
print(f"The implementation uses {used_features} features")



The implementation uses 22 features


### Feature selection - each environment

In [ ]:
#Filter for each environment

x_avenue,  y_avenue = data_avenue[feature_columns], data_avenue[qos_column]
x_park,  y_park = data_park[feature_columns], data_park[qos_column]
x_residential,  y_residential = data_residential[feature_columns], data_residential[qos_column]
x_highway,  y_highway = data_highway[feature_columns], data_highway[qos_column]
x_tunnel, y_tunnel = data_tunnel[feature_columns], data_tunnel[qos_column]

x_avenue = x_avenue.fillna(0)
x_park = x_park.fillna(0)
x_residential = x_residential.fillna(0)
x_highway = x_highway.fillna(0)
x_tunnel = x_tunnel.fillna(0)


<ipython-input-7-0101d9e2cfb9>:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x_tunnel = x_tunnel.fillna(0)


In [ ]:
#Separating data in Training, validation and test
from sklearn.model_selection import train_test_split
train_ratio = 0.80
validation_ratio = 0.10
test_ratio = 0.10


#Avenue
x_traina,x_testav,y_traina,y_testav=train_test_split(x_avenue,y_avenue,test_size=1 - train_ratio,random_state=0)
x_vala,x_testa,y_vala,y_testa=train_test_split(x_testav,y_testav,test_size=test_ratio/(test_ratio + validation_ratio),random_state=0)


#Residential
x_trainr,x_testrv,y_trainr,y_testrv=train_test_split(x_residential,y_residential,test_size=1 - train_ratio,random_state=0)
x_valr,x_testr,y_valr,y_testr=train_test_split(x_testrv,y_testrv,test_size=test_ratio/(test_ratio + validation_ratio),random_state=0)


#Park
x_trainp,x_testpv,y_trainp,y_testpv=train_test_split(x_park,y_park,test_size=1 - train_ratio,random_state=0)
x_valp,x_testp,y_valp,y_testp=train_test_split(x_testpv,y_testpv,test_size=test_ratio/(test_ratio + validation_ratio),random_state=0)

#Highway
x_trainh,x_testhv,y_trainh,y_testhv=train_test_split(x_highway,y_highway,test_size=1 - train_ratio,random_state=0)
x_valh,x_testh,y_valh,y_testh=train_test_split(x_testhv,y_testhv,test_size=test_ratio/(test_ratio + validation_ratio),random_state=0)

#Tunnel
x_traint,x_testtv,y_traint,y_testtv=train_test_split(x_tunnel,y_tunnel,test_size=1 - train_ratio,random_state=0)
x_valt,x_testt,y_valt,y_testt=train_test_split(x_testtv,y_testtv,test_size=test_ratio/(test_ratio + validation_ratio),random_state=0)

#Total data - separated per environment

x_trainarea=pd.concat([x_traina,x_trainp,x_trainr,x_trainh,x_traint])
print(x_trainarea.shape)
x_valarea=pd.concat([x_vala,x_valp,x_valr,x_valh,x_valt])
print(x_valarea.shape)
x_testarea=pd.concat([x_testa,x_testp,x_testr,x_testh,x_testt])
print(x_testarea.shape)
y_trainarea=pd.concat([y_traina,y_trainp,y_trainr,y_trainh,y_traint])
y_valarea=pd.concat([y_vala,y_valp,y_valr,y_valh,y_valt])
y_testarea=pd.concat([y_testa,y_testp,y_testr,y_testh,y_testt])


(25646, 22)
(3207, 22)
(3207, 22)




## Part 1 - Hyperparameter Tuning for Random Forest

### Study Number of Depth

In [ ]:
#Study Number of depth
import pickle#Save models
modelspath='/content/drive/MyDrive/Doutorado - UFBA/modelos_randomforest/'
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error

ndepth=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20]
reps=5#number of repetitions
shapetest=(len(ndepth),reps)
r2_scorerf=np.zeros(shapetest)#r2_score
MAErf=np.zeros(shapetest)#Mean Absolute Error
RMSErf=np.zeros(shapetest)# Root Mean Squared Error
temptrain=np.zeros(shapetest)# Tempo de treinamento
temppred=np.zeros(shapetest)# Tempo de predição

j=0
for j in range(reps):
  print('Repetition: '+str(j+1))
  i=0
  for ndep in ndepth:
        rf = RandomForestRegressor(max_depth=ndep,max_features=int(used_features/3))
        tempo_init=(time.time())#initial training time
        rf.fit(x_trainarea, y_trainarea)#fit the RF model
        tempo_fint=(time.time())#final training time
        temptrain[i,j]=tempo_fint-tempo_init
        print(f"{temptrain[i]} segundos para treinar o modelo")

        tempo_inip=(time.time())#initial predict time
        y_pred=rf.predict(x_valarea)#predict
        tempo_finp=(time.time())#final predict time
        temppred[i,j]=tempo_finp-tempo_inip
        print(f"{temppred[i]} segundos para predizer com o modelo")

        print('Metrics for RF: '+'max_depth='+str(ndep))
        r2_scorerf[i,j]=r2_score(y_valarea,y_pred)#evaluate r2_score
        print('R2_Score='+str(r2_scorerf[i]))
        MAErf[i,j]=mean_absolute_error(y_valarea,y_pred)#evaluate MAE (validação)
        print('MAE='+str(MAErf[i]))
        RMSErf[i,j]=mean_squared_error(y_valarea,y_pred, squared=False)#evaluate RMSE (validação)
        print('RMSE='+str(RMSErf[i]))
        #Save the model with pickle
#        with open(modelspath+'ndepth_'+str(ndep)+'rep'+str(j)+'.pkl', 'wb') as f:
#          pickle.dump(rf, f)
        i=i+1



Repetition: 1
[3.15240645 0.         0.         0.         0.        ] segundos para treinar o modelo
[0.05317378 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metrics for RF: max_depth=1
R2_Score=[0.60789818 0.         0.         0.         0.        ]
MAE=[17.52990434  0.          0.          0.          0.        ]
RMSE=[24.45140441  0.          0.          0.          0.        ]
[3.86817861 0.         0.         0.         0.        ] segundos para treinar o modelo
[0.02961326 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metrics for RF: max_depth=2
R2_Score=[0.79571275 0.         0.         0.         0.        ]
MAE=[12.07990379  0.          0.          0.          0.        ]
RMSE=[17.64919945  0.          0.          0.          0.        ]
[4.18795013 0.         0.         0.         0.        ] segundos para treinar o modelo
[0.03721976 0.         0.         0.         0.        ] segundos para predizer co

In [ ]:
print("R2 Score: ")
for i in range(len(ndepth)):
  for j in range(reps):
    print(r2_scorerf[i,j])

print("MAE: ")
for i in range(len(ndepth)):
  for j in range(reps):
    print(MAErf[i,j])

print("RMSErf: ")
for i in range(len(ndepth)):
  for j in range(reps):
   print(RMSErf[i,j])

print("Tempo de Treinamento: ")
for i in range(len(ndepth)):
  for j in range(reps):
    print(temptrain[i,j])

print("Tempo de Predição: ")
for i in range(len(ndepth)):
  for j in range(reps):
    print(temppred[i,j])


R2 Score: 
0.6078981772115005
0.6189823067922315
0.6056821512449786
0.6171225905691071
0.6132978761669287
0.7957127483436317
0.7923367874530715
0.7881255121495896
0.8000952680293374
0.7903694517277685
0.866680525639528
0.8672474530075143
0.866416015968242
0.8668345696553248
0.8650316171573977
0.9024122000784692
0.905272193338412
0.9063123887201107
0.90395862449778
0.90628206325058
0.9233330421684148
0.9222952550251595
0.922121819377895
0.9253318115269775
0.923006295109483
0.9356688658516071
0.93479741958494
0.9354051710547893
0.9348552473862319
0.9351724712972959
0.9411284275782624
0.941006692855122
0.9412778170529228
0.9415075867939338
0.9411344191046388
0.94502822219168
0.9440525005540706
0.9443166935052472
0.9445173925910636
0.9439831615706329
0.9467425835560174
0.9467039531727328
0.9464400610270649
0.9465881423008377
0.9471982374333268
0.9477655298925813
0.9476275694660279
0.9480304817734442
0.9472303994839439
0.9475579382799633
0.948197254389374
0.9488885309030998
0.94876646088552

###Study Number of Estimators

In [ ]:
#Study Number of Estimators

import pickle#Save models
modelspath='/content/drive/MyDrive/Doutorado - UFBA/modelos_randomforest/'
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error

nestimators=[50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000]#number of estimators
reps=5#number of repetitions
shapetest=(len(nestimators),reps)
r2_scorerf=np.zeros(shapetest)#r2_score
MAErf=np.zeros(shapetest)#Mean Absolute Error
RMSErf=np.zeros(shapetest)# Root Mean Squared Error
temptrain=np.zeros(shapetest)# Tempo de treinamento
temppred=np.zeros(shapetest)# Tempo de predição

j=0
for j in range(reps):
  print('Repetition: '+str(j+1))
  i=0
  for nest in nestimators:
        rf = RandomForestRegressor(n_estimators=nest,max_features=int(used_features/3))
        tempo_init=(time.time())#initial training time
        rf.fit(x_trainarea, y_trainarea)#fit the RF model
        tempo_fint=(time.time())#final training time
        temptrain[i,j]=tempo_fint-tempo_init
        print(f"{temptrain[i]} segundos para treinar o modelo")

        tempo_inip=(time.time())#initial predict time
        y_pred=rf.predict(x_valarea)#predict
        tempo_finp=(time.time())#final predict time
        temppred[i,j]=tempo_finp-tempo_inip
        print(f"{temppred[i]} segundos para predizer com o modelo")


        print('Metrics for RF: '+'nestimators='+str(nest))
        r2_scorerf[i,j]=r2_score(y_valarea,y_pred)#evaluate r2_score
        print('R2_Score='+str(r2_scorerf[i]))
        MAErf[i,j]=mean_absolute_error(y_valarea,y_pred)#evaluate MAE
        print('MAE='+str(MAErf[i]))
        RMSErf[i,j]=mean_squared_error(y_valarea,y_pred, squared=False)#evaluate RMSE
        print('RMSE='+str(RMSErf[i]))
        #Save the model with pickle
 #       with open(modelspath+'nestimators_'+str(nest)+'rep'+str(j)+'.pkl', 'wb') as f:
 #         pickle.dump(rf, f)
        i=i+1



Repetition: 1
[6.87023544 0.         0.         0.         0.        ] segundos para treinar o modelo
[0.07853246 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metrics for RF: nestimators=50
R2_Score=[0.94832596 0.         0.         0.         0.        ]
MAE=[5.11387835 0.         0.         0.         0.        ]
RMSE=[8.87647624 0.         0.         0.         0.        ]
[15.04213953  0.          0.          0.          0.        ] segundos para treinar o modelo
[0.14899683 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metrics for RF: nestimators=100
R2_Score=[0.94969211 0.         0.         0.         0.        ]
MAE=[5.05478276 0.         0.         0.         0.        ]
RMSE=[8.7583531 0.        0.        0.        0.       ]
[23.39844728  0.          0.          0.          0.        ] segundos para treinar o modelo
[0.22868085 0.         0.         0.         0.        ] segundos para predizer com o mode

In [ ]:
print("R2 Score: ")
for i in range(len(nestimators)):
  for j in range(reps):
    print(r2_scorerf[i,j])

print("MAE: ")
for i in range(len(nestimators)):
  for j in range(reps):
    print(MAErf[i,j])

print("RMSErf: ")
for i in range(len(nestimators)):
  for j in range(reps):
   print(RMSErf[i,j])

print("Tempo de Treinamento: ")
for i in range(len(nestimators)):
  for j in range(reps):
    print(temptrain[i,j])

print("Tempo de Predição: ")
for i in range(len(nestimators)):
  for j in range(reps):
    print(temppred[i,j])

R2 Score: 
0.948325957971129
0.9475947615146676
0.9480010276803543
0.9491311303691525
0.9480032197371586
0.9496921051483682
0.9496410963111698
0.948778372591772
0.9489131957962438
0.9493346581643242
0.9492125885975772
0.9495430854646666
0.9493786860737403
0.9493964975243466
0.9494554860316051
0.9493162503588897
0.9498076452438172
0.949632980919284
0.9489798331724085
0.9494974584981677
0.9491795423393219
0.9496912474628622
0.9493597119274197
0.9495397226421878
0.9494984757061163
0.9491174036629644
0.9495656999346379
0.9497713764993936
0.9498325597689454
0.9495303130746852
0.9495374714401036
0.9494817566735709
0.9497605197627568
0.949689989983691
0.9491839576905267
0.9499849813800448
0.949610921087495
0.9496722057519125
0.9496033241883173
0.9497663706223994
0.9497093635168311
0.9498812774576889
0.9496669169396205
0.9495603796314981
0.9497774772004397
0.9497303486657447
0.9497576634016459
0.94960659325333
0.949568354726102
0.9496926387646183
0.949455939153936
0.9500302617334494
0.94978139

###Study Max Features

In [ ]:
#Study Max Features

import pickle#Save models
modelspath='/content/drive/MyDrive/Doutorado - UFBA/modelos_randomforest/'
import time

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error

maxfeatures=["log2", "sqrt", 5, 6, int(used_features/3), 8, 9, 10, 11, 12,13,14,15,16,17,18,19,20,21,22]
reps=5#number of repetitions
shapetest=(len(maxfeatures),reps)
r2_scorerf=np.zeros(shapetest)#r2_score
MAErf=np.zeros(shapetest)#Mean Absolute Error
RMSErf=np.zeros(shapetest)# Root Mean Squared Error
temptrain=np.zeros(shapetest)# Tempo de treinamento
temppred=np.zeros(shapetest)# Tempo de predição


j=0
for j in range(reps):
  print('Repetition: '+str(j+1))
  i=0
  for max in maxfeatures:
        rf = RandomForestRegressor(max_features=max)
        tempo_init=(time.time())#initial training time
        rf.fit(x_trainarea, y_trainarea)#fit the RF model
        tempo_fint=(time.time())#final training time
        temptrain[i,j]=tempo_fint-tempo_init
        print(f"{temptrain[i]} segundos para treinar o modelo")

        tempo_inip=(time.time())#initial predict time
        y_pred=rf.predict(x_valarea)#predict
        tempo_finp=(time.time())#final predict time
        temppred[i,j]=tempo_finp-tempo_inip
        print(f"{temppred[i]} segundos para predizer com o modelo")


        print('Metrics for RF: '+'max_features='+str(max))
        r2_scorerf[i,j]=r2_score(y_valarea,y_pred)#evaluate r2_score
        print('R2_Score='+str(r2_scorerf[i]))
        MAErf[i,j]=mean_absolute_error(y_valarea,y_pred)#evaluate MAE
        print('MAE='+str(MAErf[i]))
        RMSErf[i,j]=mean_squared_error(y_valarea,y_pred, squared=False)#evaluate RMSE
        print('RMSE='+str(RMSErf[i]))
        #Save the model with pickle
#        with open(modelspath+'max_feat_'+str(max)+'rep'+str(j)+'.pkl', 'wb') as f:
#          pickle.dump(rf, f)
        i=i+1


Repetition: 1
[9.98414016 0.         0.         0.         0.        ] segundos para treinar o modelo
[0.17780209 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metrics for RF: max_features=log2
R2_Score=[0.94855283 0.         0.         0.         0.        ]
MAE=[5.19196504 0.         0.         0.         0.        ]
RMSE=[8.85696866 0.         0.         0.         0.        ]
[8.1646893 0.        0.        0.        0.       ] segundos para treinar o modelo
[0.14753962 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metrics for RF: max_features=sqrt
R2_Score=[0.94780534 0.         0.         0.         0.        ]
MAE=[5.23754778 0.         0.         0.         0.        ]
RMSE=[8.92107986 0.         0.         0.         0.        ]
[11.82402921  0.          0.          0.          0.        ] segundos para treinar o modelo
[0.14940429 0.         0.         0.         0.        ] segundos para predizer com o mode

In [ ]:
print("R2 Score: ")
for i in range(len(maxfeatures)):
  for j in range(reps):
    print(r2_scorerf[i,j])

print("MAE: ")
for i in range(len(maxfeatures)):
  for j in range(reps):
    print(MAErf[i,j])

print("RMSErf: ")
for i in range(len(maxfeatures)):
  for j in range(reps):
   print(RMSErf[i,j])



print("Tempo de Treinamento: ")
for i in range(len(maxfeatures)):
  for j in range(reps):
    print(temptrain[i,j])

print("Tempo de Predição: ")
for i in range(len(maxfeatures)):
  for j in range(reps):
    print(temppred[i,j])



R2 Score: 
0.9485528336230324
0.9479830776833013
0.9479183843609842
0.9480541556643832
0.948216140663724
0.947805336921634
0.9464043643209491
0.9489352763265981
0.9471076423812385
0.9480386177454979
0.9485926802162364
0.948619564025091
0.9485920468272603
0.9490442376180653
0.9482767780491632
0.9483590901086053
0.9493622430803501
0.9492532490440406
0.9487208451838057
0.9495190448344911
0.9491191266219318
0.9490665714558518
0.949324644395224
0.9489370238412281
0.9495387084973791
0.9491333186125154
0.9490480617668873
0.94895061650914
0.9488080385214519
0.9491177181262874
0.9490438550210643
0.949145829787664
0.9484209829432739
0.9490530033163095
0.9492709305458717
0.9490798970046072
0.9490381881455685
0.9492407388656782
0.9490826542376924
0.949657268996429
0.9489689654440432
0.9496194582824359
0.9491856117476007
0.949166327520949
0.9495711932403043
0.9502829592556093
0.9491620233847022
0.9489252478169787
0.9487782045040363
0.9499872558501962
0.9490807796814732
0.9486808978614015
0.94827552

###Study Max Samples

In [ ]:
#Study Max Samples

import pickle#Save models
modelspath='/content/drive/MyDrive/Doutorado - UFBA/modelos_randomforest/'
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error

maxsamples=[0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
reps=5#number of repetitions
shapetest=(len(maxsamples),reps)
r2_scorerf=np.zeros(shapetest)#r2_score
MAErf=np.zeros(shapetest)#Mean Absolute Error
RMSErf=np.zeros(shapetest)# Root Mean Squared Error
temptrain=np.zeros(shapetest)# Tempo de treinamento
temppred=np.zeros(shapetest)# Tempo de predição

j=0
for j in range(reps):
  print('Repetition: '+str(j+1))
  i=0
  for maxs in maxsamples:
        rf = RandomForestRegressor(max_samples=maxs,max_features=int(used_features/3))
        tempo_init=(time.time())#initial training time
        rf.fit(x_trainarea, y_trainarea)#fit the RF model
        tempo_fint=(time.time())#final training time
        temptrain[i,j]=tempo_fint-tempo_init
        print(f"{temptrain[i]} segundos para treinar o modelo")

        tempo_inip=(time.time())#initial predict time
        y_pred=rf.predict(x_valarea)#predict
        tempo_finp=(time.time())#final predict time
        temppred[i,j]=tempo_finp-tempo_inip
        print(f"{temppred[i]} segundos para predizer com o modelo")

        print('Metrics for RF: '+'max_samples='+str(maxs))
        r2_scorerf[i,j]=r2_score(y_valarea,y_pred)#evaluate r2_score
        print('R2_Score='+str(r2_scorerf[i]))
        MAErf[i,j]=mean_absolute_error(y_valarea,y_pred)#evaluate MAE
        print('MAE='+str(MAErf[i]))
        RMSErf[i,j]=mean_squared_error(y_valarea,y_pred, squared=False)#evaluate RMSE
        print('RMSE='+str(RMSErf[i]))

        #Save the model with pickle
#        with open(modelspath+'max_samples_'+str(maxs)+'rep'+str(j)+'.pkl', 'wb') as f:
#          pickle.dump(rf, f)
        i=i+1


Repetition: 1
[1.0607655 0.        0.        0.        0.       ] segundos para treinar o modelo
[0.06068921 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metrics for RF: max_samples=0.05
R2_Score=[0.93931917 0.         0.         0.         0.        ]
MAE=[5.53016312 0.         0.         0.         0.        ]
RMSE=[9.61900471 0.         0.         0.         0.        ]
[1.71763349 0.         0.         0.         0.        ] segundos para treinar o modelo
[0.07326913 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metrics for RF: max_samples=0.1
R2_Score=[0.94277467 0.         0.         0.         0.        ]
MAE=[5.36323572 0.         0.         0.         0.        ]
RMSE=[9.34111113 0.         0.         0.         0.        ]
[2.49608207 0.         0.         0.         0.        ] segundos para treinar o modelo
[0.08387613 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metri

In [ ]:
print("R2 Score: ")
for i in range(len(maxsamples)):
  for j in range(reps):
    print(r2_scorerf[i,j])

print("MAE: ")
for i in range(len(maxsamples)):
  for j in range(reps):
    print(MAErf[i,j])

print("RMSErf: ")
for i in range(len(maxsamples)):
  for j in range(reps):
   print(RMSErf[i,j])


print("Tempo de Treinamento: ")
for i in range(len(maxsamples)):
  for j in range(reps):
    print(temptrain[i,j])

print("Tempo de Predição: ")
for i in range(len(maxsamples)):
  for j in range(reps):
    print(temppred[i,j])

R2 Score: 
0.9393191691866352
0.9404312971297281
0.9406127443777994
0.940858986303106
0.9378316450974292
0.9427746679552266
0.9424252216410449
0.9427896621946951
0.9431901759229716
0.9436419517339347
0.9452041840005906
0.945158639022956
0.9451544598087419
0.9445218289455957
0.945242894489769
0.9457872251314005
0.9460738058639018
0.9463659988120011
0.9461428961047902
0.9462201431471373
0.9475857872157312
0.9472128389412495
0.9468934153947006
0.9470325990648857
0.9469186797418879
0.9478546379864583
0.948228576573144
0.9465907116347583
0.9477643735851724
0.9471684818895398
0.946297768337821
0.9477091962389064
0.9479516792125291
0.9474595630793692
0.9471958054702183
0.9478984813606715
0.9482171935045377
0.9478507687447255
0.9468248575453243
0.9486036618279751
0.9467877585347192
0.9481011137312289
0.947665818069543
0.9483527411815457
0.9488142130697832
0.9476923555275323
0.9487442732995419
0.948085271814734
0.9477135793474311
0.9486572849747015
0.9480184942195128
0.9485425169069619
0.948623

###Study Max Leaf Nodes

In [ ]:
#Study Max Leaf Nodes

import pickle#Save models
modelspath='/content/drive/MyDrive/Doutorado - UFBA/modelos_randomforest/'
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error

maxleafn=[50, 100, 200, 300, 400, 500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000, 6000, 7000, 8000, 9000, 10000, None]

reps=5#number of repetitions
shapetest=(len(maxleafn),reps)
r2_scorerf=np.zeros(shapetest)#r2_score
MAErf=np.zeros(shapetest)#Mean Absolute Error
RMSErf=np.zeros(shapetest)# Root Mean Squared Error

temptrain=np.zeros(shapetest)# Tempo de treinamento
temppred=np.zeros(shapetest)# Tempo de predição


j=0
for j in range(reps):
  print('Repetition: '+str(j+1))
  i=0
  for maxl in maxleafn:
        rf = RandomForestRegressor(max_leaf_nodes=maxl,max_features=int(used_features/3))
        tempo_init=(time.time())#initial training time
        rf.fit(x_trainarea, y_trainarea)#fit the RF model
        tempo_fint=(time.time())#final training time
        temptrain[i,j]=tempo_fint-tempo_init
        print(f"{temptrain[i]} segundos para treinar o modelo")

        tempo_inip=(time.time())#initial predict time
        y_pred=rf.predict(x_valarea)#predict
        tempo_finp=(time.time())#final predict time
        temppred[i,j]=tempo_finp-tempo_inip
        print(f"{temppred[i]} segundos para predizer com o modelo")

        print('Metrics for RF: '+'max_leafnodes='+str(maxl))
        r2_scorerf[i,j]=r2_score(y_valarea,y_pred)#evaluate r2_score
        print('R2_Score='+str(r2_scorerf[i]))
        MAErf[i,j]=mean_absolute_error(y_valarea,y_pred)#evaluate MAE
        print('MAE='+str(MAErf[i]))
        RMSErf[i,j]=mean_squared_error(y_valarea,y_pred, squared=False)#evaluate RMSE
        print('RMSE='+str(RMSErf[i]))

        #Save the model with pickle
#        with open(modelspath+'max_leaf_nodes'+str(maxl)+'rep'+str(j)+'.pkl', 'wb') as f:
#          pickle.dump(rf, f)
        i=i+1


Repetition: 1
[5.58198309 0.         0.         0.         0.        ] segundos para treinar o modelo
[0.02732921 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metrics for RF: max_leafnodes=50
R2_Score=[0.93665786 0.         0.         0.         0.        ]
MAE=[6.02511473 0.         0.         0.         0.        ]
RMSE=[9.8276738 0.        0.        0.        0.       ]
[7.81479955 0.         0.         0.         0.        ] segundos para treinar o modelo
[0.04442716 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metrics for RF: max_leafnodes=100
R2_Score=[0.94221141 0.         0.         0.         0.        ]
MAE=[5.65109937 0.         0.         0.         0.        ]
RMSE=[9.38697 0.      0.      0.      0.     ]
[6.99269509 0.         0.         0.         0.        ] segundos para treinar o modelo
[0.0345335 0.        0.        0.        0.       ] segundos para predizer com o modelo
Metrics for RF: max_lea

In [ ]:
print("R2 Score: ")
for i in range(len(maxleafn)):
  for j in range(reps):
    print(r2_scorerf[i,j])

print("MAE: ")
for i in range(len(maxleafn)):
  for j in range(reps):
    print(MAErf[i,j])

print("RMSErf: ")
for i in range(len(maxleafn)):
  for j in range(reps):
   print(RMSErf[i,j])

print("Tempo de Treinamento: ")
for i in range(len(maxleafn)):
  for j in range(reps):
    print(temptrain[i,j])

print("Tempo de Predição: ")
for i in range(len(maxleafn)):
  for j in range(reps):
    print(temppred[i,j])


R2 Score: 
0.9366578631745357
0.9362308865688709
0.9358779607662533
0.9365884935644433
0.9370567236181819
0.9422114092761913
0.9430841502103237
0.9432383556184494
0.9425091986876024
0.9417965173253384
0.9459690765180079
0.9466532259276873
0.9455129517113141
0.9462065679509611
0.945921830477037
0.947412099991247
0.9473919258613739
0.947074027180673
0.9469886962853038
0.9478356070001522
0.9470554403561433
0.9490413382921221
0.947710503216776
0.9475592905648723
0.9479478772846546
0.9479425700611089
0.9471575134097896
0.9477854148565004
0.9477391782852254
0.9480343666888738
0.948975082968516
0.9491760497757395
0.9488552589587477
0.9488783831226877
0.949331709873923
0.948954476315364
0.9488992794010429
0.9495888569398835
0.9498016490426553
0.9493139460831771
0.949210560346708
0.9489098661653383
0.9485663874945033
0.9488265218229377
0.9487762738472213
0.9494665749350159
0.949198120681304
0.9495370481064479
0.9490932938595633
0.948013763366264
0.9495766140416704
0.9486104192909085
0.949459983

###Study Min Samples Leaf

In [ ]:
#Study Min Samples Leaf

import pickle#Save models
modelspath='/content/drive/MyDrive/Doutorado - UFBA/modelos_randomforest/'
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error

minsamplesleaf=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

reps=5#number of repetitions
shapetest=(len(minsamplesleaf),reps)
r2_scorerf=np.zeros(shapetest)#r2_score
MAErf=np.zeros(shapetest)#Mean Absolute Error
RMSErf=np.zeros(shapetest)# Root Mean Squared Error

temptrain=np.zeros(shapetest)# Tempo de treinamento
temppred=np.zeros(shapetest)# Tempo de predição


j=0
for j in range(reps):
  print('Repetition: '+str(j+1))
  i=0
  for minsamples in minsamplesleaf:
        rf = RandomForestRegressor(min_samples_leaf=minsamples,max_features=int(used_features/3))
        tempo_init=(time.time())#initial training time
        rf.fit(x_trainarea, y_trainarea)#fit the RF model
        tempo_fint=(time.time())#final training time
        temptrain[i,j]=tempo_fint-tempo_init
        print(f"{temptrain[i]} segundos para treinar o modelo")

        tempo_inip=(time.time())#initial predict time
        y_pred=rf.predict(x_valarea)#predict
        tempo_finp=(time.time())#final predict time
        temppred[i,j]=tempo_finp-tempo_inip
        print(f"{temppred[i]} segundos para predizer com o modelo")


        print('Metrics for RF: '+'minsamplesleaf='+str(minsamples))
        r2_scorerf[i,j]=r2_score(y_valarea,y_pred)#evaluate r2_score
        print('R2_Score='+str(r2_scorerf[i]))
        MAErf[i,j]=mean_absolute_error(y_valarea,y_pred)#evaluate MAE
        print('MAE='+str(MAErf[i]))
        RMSErf[i,j]=mean_squared_error(y_valarea,y_pred, squared=False)#evaluate RMSE
        print('RMSE='+str(RMSErf[i]))

        #Save the model with pickle
    #    with open(modelspath+'min_sample_leafs'+str(minsamples)+'rep'+str(j)+'.pkl', 'wb') as f:
    #      pickle.dump(rf, f)
        i=i+1


Repetition: 1
[15.33801007  0.          0.          0.          0.        ] segundos para treinar o modelo
[0.15184999 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metrics for RF: minsamplesleaf=1
R2_Score=[0.94941655 0.         0.         0.         0.        ]
MAE=[5.0713137 0.        0.        0.        0.       ]
RMSE=[8.78230651 0.         0.         0.         0.        ]
[13.20740628  0.          0.          0.          0.        ] segundos para treinar o modelo
[0.10980415 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metrics for RF: minsamplesleaf=2
R2_Score=[0.94907811 0.         0.         0.         0.        ]
MAE=[5.05559848 0.         0.         0.         0.        ]
RMSE=[8.81163814 0.         0.         0.         0.        ]
[12.42380238  0.          0.          0.          0.        ] segundos para treinar o modelo
[0.0935967 0.        0.        0.        0.       ] segundos para predizer com o m

In [ ]:
print("R2 Score: ")
for i in range(len(minsamplesleaf)):
  for j in range(reps):
    print(r2_scorerf[i,j])

print("MAE: ")
for i in range(len(minsamplesleaf)):
  for j in range(reps):
    print(MAErf[i,j])

print("RMSErf: ")
for i in range(len(minsamplesleaf)):
  for j in range(reps):
   print(RMSErf[i,j])

print("Tempo de Treinamento: ")
for i in range(len(minsamplesleaf)):
  for j in range(reps):
    print(temptrain[i,j])

print("Tempo de Predição: ")
for i in range(len(minsamplesleaf)):
  for j in range(reps):
    print(temppred[i,j])


R2 Score: 
0.9494165526160372
0.9487247186663571
0.9488708170064323
0.9497356685509993
0.9501825314432794
0.9490781056261297
0.9484333701103587
0.9494141805189604
0.9491379704898896
0.9494657284298433
0.9491686002992049
0.9490104030614279
0.9490196483096336
0.9487350132859937
0.949330089755519
0.9488954089035384
0.9489526302950786
0.9489633860807356
0.9491240523015922
0.9488314219238041
0.9488808357439895
0.9486754671265392
0.9478892830417459
0.9485423432571712
0.9490099686811515
0.9485612477183014
0.9486148448776895
0.9491377081652473
0.9479441607741759
0.9484076275431196
0.947787436088504
0.9478138201562034
0.9476561770530426
0.9483457599112213
0.9480091970905865
0.9478267500326082
0.9476622097083897
0.9478209668418776
0.947647423260354
0.9476144765617762
0.947582472629881
0.9472663100800492
0.9469913455798934
0.9473928136246365
0.9471061659842972
0.9471311102939239
0.9474178454909282
0.9472597666194741
0.9470148416444638
0.9471300903793914
0.9464027126868755
0.9469872996004863
0.946

###Study Min Samples Split

In [ ]:
#Study Min Samples Split

import pickle#Save models
modelspath='/content/drive/MyDrive/Doutorado - UFBA/modelos_randomforest/'
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error

minsamplessplit=[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]

reps=5#number of repetitions
shapetest=(len(minsamplessplit),reps)
r2_scorerf=np.zeros(shapetest)#r2_score
MAErf=np.zeros(shapetest)#Mean Absolute Error
RMSErf=np.zeros(shapetest)# Root Mean Squared Error
temptrain=np.zeros(shapetest)# Tempo de treinamento
temppred=np.zeros(shapetest)# Tempo de predição

j=0
for j in range(reps):
  print('Repetition: '+str(j+1))
  i=0
  for minss in minsamplessplit:
        rf = RandomForestRegressor(min_samples_split=minss,max_features=int(used_features/3))
        tempo_init=(time.time())#initial training time
        rf.fit(x_trainarea, y_trainarea)#fit the RF model
        tempo_fint=(time.time())#final training time
        temptrain[i,j]=tempo_fint-tempo_init
        print(f"{temptrain[i]} segundos para treinar o modelo")

        tempo_inip=(time.time())#initial predict time
        y_pred=rf.predict(x_valarea)#predict
        tempo_finp=(time.time())#final predict time
        temppred[i,j]=tempo_finp-tempo_inip
        print(f"{temppred[i]} segundos para predizer com o modelo")


        print('Metrics for RF: '+'minsamplesplit='+str(minss))
        r2_scorerf[i,j]=r2_score(y_valarea,y_pred)#evaluate r2_score
        print('R2_Score='+str(r2_scorerf[i]))
        MAErf[i,j]=mean_absolute_error(y_valarea,y_pred)#evaluate MAE
        print('MAE='+str(MAErf[i]))
        RMSErf[i,j]=mean_squared_error(y_valarea,y_pred, squared=False)#evaluate RMSE
        print('RMSE='+str(RMSErf[i]))

        #Save the model with pickle
#        with open(modelspath+'min_sample_split'+str(minss)+'rep'+str(j)+'.pkl', 'wb') as f:
#          pickle.dump(rf, f)
        i=i+1


Repetition: 1
[10.28169274  0.          0.          0.          0.        ] segundos para treinar o modelo
[0.11439347 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metrics for RF: minsamplesplit=2
R2_Score=[0.94981901 0.         0.         0.         0.        ]
MAE=[5.05791942 0.         0.         0.         0.        ]
RMSE=[8.74729922 0.         0.         0.         0.        ]
[9.86280894 0.         0.         0.         0.        ] segundos para treinar o modelo
[0.09870052 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metrics for RF: minsamplesplit=3
R2_Score=[0.94910523 0.         0.         0.         0.        ]
MAE=[5.06480335 0.         0.         0.         0.        ]
RMSE=[8.80929096 0.         0.         0.         0.        ]
[9.83178282 0.         0.         0.         0.        ] segundos para treinar o modelo
[0.09061146 0.         0.         0.         0.        ] segundos para predizer com o m

In [ ]:
print("R2 Score: ")
for i in range(len(minsamplessplit)):
  for j in range(reps):
    print(r2_scorerf[i,j])

print("MAE: ")
for i in range(len(minsamplessplit)):
  for j in range(reps):
    print(MAErf[i,j])

print("RMSErf: ")
for i in range(len(minsamplessplit)):
  for j in range(reps):
   print(RMSErf[i,j])

print("Tempo de Treinamento: ")
for i in range(len(minsamplessplit)):
  for j in range(reps):
    print(temptrain[i,j])

print("Tempo de Predição: ")
for i in range(len(minsamplessplit)):
  for j in range(reps):
    print(temppred[i,j])



R2 Score: 
0.9498190117747691
0.9494408346555018
0.9488660053060473
0.9494510089224335
0.9489817566756977
0.9491052304463821
0.9484464423051812
0.9492145055091228
0.9491523486234674
0.9495843895080018
0.9491777264298439
0.9492005729365999
0.949399037738681
0.9485600799659811
0.949227249097387
0.9496308349177636
0.9487656275813565
0.9484580162754059
0.9489781537559243
0.9488526269912314
0.9494678192709621
0.9492735112530551
0.94905261271933
0.9486003126592126
0.9488255489773227
0.9494274806325433
0.9494371079396461
0.9481187148268805
0.9495265085929175
0.9485402950366605
0.9487185367949653
0.9488427504294104
0.9485595524243655
0.9481491968151717
0.9493171636433755
0.9492927006973981
0.9493424618735103
0.9491779554899348
0.9496874214898692
0.9492193279671007
0.9494388760427102
0.9491491401747563
0.9495325793992035
0.9490015537547001
0.9493703633130989
0.9491278818795407
0.948358751227011
0.9489209371680251
0.9486764832784416
0.9488778834215484
0.9493882803785928
0.9497239616951849
0.9495

###Study Criterion

In [ ]:
#Study Criterion

import pickle#Save models
modelspath='/content/drive/MyDrive/Doutorado - UFBA/modelos_randomforest/'
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error

criteri=["squared_error", "absolute_error", "friedman_mse", "poisson"]
reps=5#number of repetitions
shapetest=(len(criteri),reps)
r2_scorerf=np.zeros(shapetest)#r2_score
MAErf=np.zeros(shapetest)#Mean Absolute Error
RMSErf=np.zeros(shapetest)# Root Mean Squared Error
temptrain=np.zeros(shapetest)# Tempo de treinamento
temppred=np.zeros(shapetest)# Tempo de predição


j=0
for j in range(reps):
  print('Repetition: '+str(j+1))
  i=0
  for cr in criteri:
        rf = RandomForestRegressor(criterion=cr,max_features=int(used_features/3))
        tempo_init=(time.time())#initial training time
        rf.fit(x_trainarea, y_trainarea)#fit the RF model
        tempo_fint=(time.time())#final training time
        temptrain[i,j]=tempo_fint-tempo_init
        print(f"{temptrain[i]} segundos para treinar o modelo")

        tempo_inip=(time.time())#initial predict time
        y_pred=rf.predict(x_valarea)#predict
        tempo_finp=(time.time())#final predict time
        temppred[i,j]=tempo_finp-tempo_inip
        print(f"{temppred[i]} segundos para predizer com o modelo")


        print('Metrics for RF: '+'criterion='+str(cr))
        r2_scorerf[i,j]=r2_score(y_valarea,y_pred)#evaluate r2_score
        print('R2_Score='+str(r2_scorerf[i]))
        MAErf[i,j]=mean_absolute_error(y_valarea,y_pred)#evaluate MAE
        print('MAE='+str(MAErf[i]))
        RMSErf[i,j]=mean_squared_error(y_valarea,y_pred, squared=False)#evaluate RMSE
        print('RMSE='+str(RMSErf[i]))


        #Save the model with pickle
#        with open(modelspath+'criterion'+str(cr)+'rep'+str(j)+'.pkl', 'wb') as f:
#          pickle.dump(rf, f)
        i=i+1


Repetition: 1
[10.12571311  0.          0.          0.          0.        ] segundos para treinar o modelo
[0.11839771 0.         0.         0.         0.        ] segundos para predizer com o modelo
Metrics for RF: criterion=squared_error
R2_Score=[0.94897673 0.         0.         0.         0.        ]
MAE=[5.0884613 0.        0.        0.        0.       ]
RMSE=[8.82040513 0.         0.         0.         0.        ]
[745.05252171   0.           0.           0.           0.        ] segundos para treinar o modelo
[0.1162076 0.        0.        0.        0.       ] segundos para predizer com o modelo
Metrics for RF: criterion=absolute_error
R2_Score=[0.94898248 0.         0.         0.         0.        ]
MAE=[5.08399026 0.         0.         0.         0.        ]
RMSE=[8.81990816 0.         0.         0.         0.        ]
[10.03183317  0.          0.          0.          0.        ] segundos para treinar o modelo
[0.11373615 0.         0.         0.         0.        ] segundos p

In [ ]:
print("R2 Score: ")
for i in range(len(criteri)):
  for j in range(reps):
    print(r2_scorerf[i,j])

print("MAE: ")
for i in range(len(criteri)):
  for j in range(reps):
    print(MAErf[i,j])

print("RMSErf: ")
for i in range(len(criteri)):
  for j in range(reps):
   print(RMSErf[i,j])

print("Tempo de Treinamento: ")
for i in range(len(criteri)):
  for j in range(reps):
    print(temptrain[i,j])

print("Tempo de Predição: ")
for i in range(len(criteri)):
  for j in range(reps):
    print(temppred[i,j])


R2 Score: 
0.948976727387029
0.9478493871148422
0.9488767427933745
0.9487886721812224
0.9484190551718138
0.9489824768668604
0.9493234632154323
0.9499730447504225
0.9489593901531737
0.9481737717658544
0.9490015455495888
0.9498986511467928
0.9491134612030194
0.9488651582057527
0.9493448145799074
0.9487720131110933
0.9492891689303712
0.9493049933566785
0.9490563363889397
0.9482832055117603
MAE: 
5.088461304418225
5.108867820411081
5.068774814189236
5.105277342619829
5.077245053734807
5.083990255690677
5.072020037418148
5.055174972871843
5.095248096663548
5.1459208057374495
5.085069535494079
5.044184163219546
5.07496888759546
5.09988892584795
5.083905689196911
5.071327823054322
5.051368602083322
5.056432389475759
5.066632517021096
5.102898721621674
RMSErf: 
8.820405134593532
8.917314553715688
8.82904308506662
8.836644758685491
8.868476602624222
8.819908163627815
8.790383892158328
8.733863776106576
8.821903553163853
8.889537773949396
8.818259712875047
8.740355298751675
8.808578602535942
8.8

###Study Top Model

In [ ]:
import pickle#Save models
modelspath='/content/drive/MyDrive/Doutorado - UFBA/modelos_randomforest/'
import time
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error



rf = RandomForestRegressor(n_estimators=100,
                           max_features=11,
                          criterion='squared_error',
                          max_depth=20,
                           min_samples_split=2,
                           min_samples_leaf=1,
                           max_leaf_nodes=None,
                           max_samples=1.0
                               )

In [ ]:
#Study Top Model - along operators

tempo_init=(time.time())#initial training time
rf.fit(x_train, y_train)#fit the RF model
tempo_fint=(time.time())#final training time
temptrain=tempo_fint-tempo_init
print(f"{temptrain} segundos para treinar o modelo")

tempo_inip=(time.time())#initial predict time
y_pred=rf.predict(x_test)#predict in test
tempo_finp=(time.time())#final predict time
temppred=tempo_finp-tempo_inip
print(f"{temppred} segundos para predizer com o modelo")

print('Metrics for RF:')
r2_scorerf=r2_score(y_test,y_pred)#evaluate r2_score
print('R2_Score='+str(r2_scorerf))
MAErf=mean_absolute_error(y_test, y_pred)#evaluate MAE
print('MAE='+str(MAErf))
RMSErf=mean_squared_error(y_test, y_pred, squared=False)#evaluate RMSE
print('RMSE='+str(RMSErf))


#Save the model with pickle
#with open(modelspath+'topmodel.pkl', 'wb') as f:
#  pickle.dump(rf, f)

from pprint import pprint
# Look at parameters used by our current forest
print('Parameters currently in use:\n')
pprint(rf.get_params())

In [ ]:
#Study Top Model - along areas


tempo_init=(time.time())#initial training time
rf.fit(x_trainarea, y_trainarea)#fit the RF model
tempo_fint=(time.time())#final training time
temptrain=tempo_fint-tempo_init
print(f"{temptrain} segundos para treinar o modelo")


#Validation

tempo_inip=(time.time())#initial predict time
y_pred=rf.predict(x_valarea)#predict in test
tempo_finp=(time.time())#final predict time
temppred=tempo_finp-tempo_inip
print(f"{temppred} segundos para predizer com o modelo")

print('Metrics for RF (validation):')
r2_scorerf=r2_score(y_valarea,y_pred)#evaluate r2_score
print('R2_Score='+str(r2_scorerf))
MAErf=mean_absolute_error(y_valarea, y_pred)#evaluate MAE
print('MAE='+str(MAErf))
RMSErf=mean_squared_error(y_valarea, y_pred, squared=False)#evaluate RMSE
print('RMSE='+str(RMSErf))






#Testing
tempo_inip=(time.time())#initial predict time
y_pred=rf.predict(x_testarea)#predict in test
tempo_finp=(time.time())#final predict time
temppred=tempo_finp-tempo_inip
print(f"{temppred} segundos para predizer com o modelo")

print('Metrics for RF (test):')
r2_scorerf=r2_score(y_testarea,y_pred)#evaluate r2_score
print('R2_Score='+str(r2_scorerf))
MAErf=mean_absolute_error(y_testarea, y_pred)#evaluate MAE
print('MAE='+str(MAErf))
RMSErf=mean_squared_error(y_testarea, y_pred, squared=False)#evaluate RMSE
print('RMSE='+str(RMSErf))


#Save the model with pickle
#with open(modelspath+'topmodel.pkl', 'wb') as f:
#  pickle.dump(rf, f)

from pprint import pprint
# Look at parameters used by our current forest
print('Parameters currently in use:\n')
pprint(rf.get_params())

## Parte 2- Combined Hyperparameters - RandomForest Algorithm

In [ ]:
#First 2 reps
#Study Combination - max_depth+max_samples+max_leaf_nodes
import pickle#Save models
modelspath='/content/drive/MyDrive/Doutorado - UFBA/modelos_randomforest/'
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error


ndepth=[5, 10, 15, 16]
maxsamples=[0.25, 0.5, 0.7, 0.85]
maxleafn=[400, 1500, 3000, 8000]

reps=2#number of repetitions
shapetest=(len(ndepth)*len(maxsamples)*len(maxleafn),reps)
r2_scorerf=np.zeros(shapetest)#r2_score
MAErf=np.zeros(shapetest)#Mean Absolute Error
RMSErf=np.zeros(shapetest)# Root Mean Squared Error
oob_scorerf=np.zeros(shapetest)# OOB Score
temptrain=np.zeros(shapetest)# Tempo de treinamento
temppred=np.zeros(shapetest)# Tempo de predição


j=0

for j in range(reps):
  print('Repetition: '+str(j+1))
  i=0
  for ndep in ndepth:
    for maxsamp in maxsamples:
      for maxlf in maxleafn:
        rf = RandomForestRegressor(max_depth=ndep,max_samples=maxsamp,max_leaf_nodes=maxlf,n_estimators=800,
                                   max_features=12,min_samples_split=9,criterion='friedman_mse',
                                   min_samples_leaf=1)
        print('Metrics for RF: '+'max_depth='+str(ndep)+'  max_samples='+str(maxsamp)+'  max_leaf_nodes='+str(maxlf))
        tempo_init=(time.time())#initial training time
        rf.fit(x_trainarea, y_trainarea)#fit the RF model
        tempo_fint=(time.time())#final training time
        temptrain[i,j]=tempo_fint-tempo_init
        print(f"{temptrain[i,j]} segundos para treinar o modelo")

        tempo_inip=(time.time())#initial predict time
        y_pred=rf.predict(x_valarea)#predict
        tempo_finp=(time.time())#final predict time
        temppred[i,j]=tempo_finp-tempo_inip
        print(f"{temppred[i,j]} segundos para predizer com o modelo")

        r2_scorerf[i,j]=r2_score(y_valarea,y_pred)#evaluate r2_score
        print('R2_Score='+str(r2_scorerf[i,j]))
        MAErf[i,j]=mean_absolute_error(y_valarea, y_pred)#evaluate MAE
        print('MAE='+str(MAErf[i,j]))
        RMSErf[i,j]=mean_squared_error(y_valarea, y_pred, squared=False)#evaluate RMSE
        print('RMSE='+str(RMSErf[i,j]))

        i=i+1



Repetition: 1
Metrics for RF: max_depth=5  max_samples=0.25  max_leaf_nodes=400
32.521809101104736 segundos para treinar o modelo
0.15611720085144043 segundos para predizer com o modelo
R2_Score=0.9357966062628645
MAE=5.944421916821463
RMSE=9.894261187412848
Metrics for RF: max_depth=5  max_samples=0.25  max_leaf_nodes=1500


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


26.8466854095459 segundos para treinar o modelo
0.15824437141418457 segundos para predizer com o modelo
R2_Score=0.9357779527028974
MAE=5.940596253810734
RMSE=9.89569841510374
Metrics for RF: max_depth=5  max_samples=0.25  max_leaf_nodes=3000


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


25.718215942382812 segundos para treinar o modelo
0.23430633544921875 segundos para predizer com o modelo
R2_Score=0.9357751794968607
MAE=5.952234494466617
RMSE=9.895912068478928
Metrics for RF: max_depth=5  max_samples=0.25  max_leaf_nodes=8000


...

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


171.12083220481873 segundos para treinar o modelo
0.6144604682922363 segundos para predizer com o modelo
R2_Score=0.949907910948053
MAE=4.95840400175995
RMSE=8.739547559092069
Metrics for RF: max_depth=16  max_samples=0.85  max_leaf_nodes=8000


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


171.73225617408752 segundos para treinar o modelo
0.6166894435882568 segundos para predizer com o modelo
R2_Score=0.9494193715197345
MAE=4.972458285907965
RMSE=8.782061795435924


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [ ]:
print("R2 Score: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(r2_scorerf[i,j])

print("MAE: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(MAErf[i,j])

print("RMSErf: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
   print(RMSErf[i,j])


print("Tempo de Treinamento: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(temptrain[i,j])

print("Tempo de Predição: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(temppred[i,j])

R2 Score: 
repetition0
0.9357966062628645
0.9357779527028974
0.9357751794968607
0.9351989984993285
0.935498275485857
0.9353280827743461
0.9352867359338146
0.9356118297555371
0.9351456037958122
0.935189419874008
0.9347971710676577
0.9351660409801832
0.9352053341641604
0.9351543566047307
0.9356704716990979
0.9354208334611436
0.9476370997036733
0.9474320324343921
0.9475276425816178
0.94727812089994
0.9486196076694502
0.9485814164583073
0.9486738926697241
0.9486082420770102
0.9485520084827002
0.9487807729005936
0.9490123933104255
0.9487022775780458
0.9487806201659083
0.9491277741962613
0.9491050639734188
0.9490558701501034
0.9476919414785163
0.9481289690624318
0.947693310194803
0.9477524014563125
0.9484972933429596
0.9492858815615862
0.9494338967471575
0.9490687609476826
0.9483154547337969
0.9494055340132685
0.9492614674359977
0.9493810629959498
0.9488535462507885
0.949502687017228
0.9496657063074561
0.9495253745045209
0.9475086980710046
0.9477333611005747
0.9480283895067639
0.948092605110

In [ ]:
#Last 3 reps
#Study Combination - max_depth+max_samples+max_leaf_nodes
import pickle#Save models
modelspath='/content/drive/MyDrive/Doutorado - UFBA/modelos_randomforest/'
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error


ndepth=[5, 10, 15, 16]
maxsamples=[0.25, 0.5, 0.7, 0.85]
maxleafn=[400, 1500, 3000, 8000]

reps=3#number of repetitions
shapetest=(len(ndepth)*len(maxsamples)*len(maxleafn),reps)
r2_scorerf=np.zeros(shapetest)#r2_score
MAErf=np.zeros(shapetest)#Mean Absolute Error
RMSErf=np.zeros(shapetest)# Root Mean Squared Error
oob_scorerf=np.zeros(shapetest)# OOB Score
temptrain=np.zeros(shapetest)# Tempo de treinamento
temppred=np.zeros(shapetest)# Tempo de predição


j=0

for j in range(reps):
  print('Repetition: '+str(j+1))
  i=0
  for ndep in ndepth:
    for maxsamp in maxsamples:
      for maxlf in maxleafn:
        rf = RandomForestRegressor(max_depth=ndep,max_samples=maxsamp,max_leaf_nodes=maxlf,n_estimators=800,
                                   max_features=12,min_samples_split=9,criterion='friedman_mse',
                                   min_samples_leaf=1)
        print('Metrics for RF: '+'max_depth='+str(ndep)+'  max_samples='+str(maxsamp)+'  max_leaf_nodes='+str(maxlf))
        tempo_init=(time.time())#initial training time
        rf.fit(x_trainarea, y_trainarea)#fit the RF model
        tempo_fint=(time.time())#final training time
        temptrain[i,j]=tempo_fint-tempo_init
        print(f"{temptrain[i,j]} segundos para treinar o modelo")

        tempo_inip=(time.time())#initial predict time
        y_pred=rf.predict(x_valarea)#predict
        tempo_finp=(time.time())#final predict time
        temppred[i,j]=tempo_finp-tempo_inip
        print(f"{temppred[i,j]} segundos para predizer com o modelo")

        r2_scorerf[i,j]=r2_score(y_valarea,y_pred)#evaluate r2_score
        print('R2_Score='+str(r2_scorerf[i,j]))
        MAErf[i,j]=mean_absolute_error(y_valarea, y_pred)#evaluate MAE
        print('MAE='+str(MAErf[i,j]))
        RMSErf[i,j]=mean_squared_error(y_valarea, y_pred, squared=False)#evaluate RMSE
        print('RMSE='+str(RMSErf[i,j]))

        i=i+1



Repetition: 1
Metrics for RF: max_depth=5  max_samples=0.25  max_leaf_nodes=400
31.53345274925232 segundos para treinar o modelo
0.22273707389831543 segundos para predizer com o modelo
R2_Score=0.9357905398864564
MAE=5.9500052771347365
RMSE=9.894728615156044
Metrics for RF: max_depth=5  max_samples=0.25  max_leaf_nodes=1500


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


26.718073844909668 segundos para treinar o modelo
0.21878695487976074 segundos para predizer com o modelo
R2_Score=0.9353410684416748
MAE=5.944128139903046
RMSE=9.929300172399888
Metrics for RF: max_depth=5  max_samples=0.25  max_leaf_nodes=3000


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


27.773837327957153 segundos para treinar o modelo
0.2250688076019287 segundos para predizer com o modelo
R2_Score=0.9356628407739754
MAE=5.95799115738769
RMSE=9.904562995099882
Metrics for RF: max_depth=5  max_samples=0.25  max_leaf_nodes=8000


...

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


178.6123561859131 segundos para treinar o modelo
0.7594113349914551 segundos para predizer com o modelo
R2_Score=0.949344628796898
MAE=4.967663073476435
RMSE=8.788548002917945
Metrics for RF: max_depth=16  max_samples=0.85  max_leaf_nodes=8000


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


182.3207926750183 segundos para treinar o modelo
0.6722807884216309 segundos para predizer com o modelo
R2_Score=0.9495278012641089
MAE=4.968688341313144
RMSE=8.772643687954709


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [ ]:
print("R2 Score: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(r2_scorerf[i,j])

print("MAE: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(MAErf[i,j])

print("RMSErf: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
   print(RMSErf[i,j])


print("Tempo de Treinamento: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(temptrain[i,j])

print("Tempo de Predição: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(temppred[i,j])

R2 Score: 
repetition0
0.9357905398864564
0.9353410684416748
0.9356628407739754
0.9355216574693809
0.9354834909701356
0.9354606788268369
0.9353884213004517
0.9352469755877756
0.935006722271375
0.9351629797441555
0.9353989118852981
0.93527727729305
0.9351600964396425
0.9353185380842262
0.9350291597286601
0.9351458494653536
0.9475007159147825
0.9475327820655053
0.9478335069098729
0.9476812675638818
0.9483432505548589
0.9485263649885176
0.9483146965414768
0.9484927403801632
0.948654447308215
0.9489668166227079
0.9489931063101702
0.9488471275552269
0.9490511020750175
0.9489872614260296
0.9491638510816116
0.9488846553738995
0.9477726314209609
0.9477577918497266
0.9480001607601936
0.9480642966143077
0.9483093249265061
0.9489738632364548
0.9489752019848238
0.9488751520143125
0.9488490293261782
0.9492908351583965
0.9493574368932914
0.9494264460180274
0.9489663856249003
0.9497884623554329
0.9496251200132768
0.9497376321211914
0.9475320996239763
0.9478176416758131
0.9477447909147518
0.9478517443

## Parte 3- Combinations applied to the test subset

### ANOVA COMB - piores valores (grupo a)

In [ ]:
#Study Combination - max_depth+max_samples+max_leaf_nodes - piores resultados (combs - )
import pickle#Save models
modelspath='/content/drive/MyDrive/Doutorado - UFBA/modelos_randomforest/'
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error


ndepth=[5]
maxsamples=[0.85, 0.7, 0.5]
maxleafn=[8000, 400, 1500, 3000]

reps=5#number of repetitions
shapetest=(len(ndepth)*len(maxsamples)*len(maxleafn),reps)
r2_scorerf=np.zeros(shapetest)#r2_score
MAErf=np.zeros(shapetest)#Mean Absolute Error
RMSErf=np.zeros(shapetest)# Root Mean Squared Error
oob_scorerf=np.zeros(shapetest)# OOB Score
temptrain=np.zeros(shapetest)# Tempo de treinamento
temppred=np.zeros(shapetest)# Tempo de predição

j=0

for j in range(reps):
  print('Repetition: '+str(j+1))
  i=0
  for ndep in ndepth:
    for maxsamp in maxsamples:
      for maxlf in maxleafn:
        rf = RandomForestRegressor(max_depth=ndep,max_samples=maxsamp,max_leaf_nodes=maxlf,n_estimators=800,
                                   max_features=12,min_samples_split=9,criterion='friedman_mse',
                                   min_samples_leaf=1)
        print('Metrics for RF: '+'max_depth='+str(ndep)+'  max_samples='+str(maxsamp)+'  max_leaf_nodes='+str(maxlf))
        tempo_init=(time.time())#initial training time
        rf.fit(x_trainarea, y_trainarea)#fit the RF model
        tempo_fint=(time.time())#final training time
        temptrain[i,j]=tempo_fint-tempo_init
        print(f"{temptrain[i,j]} segundos para treinar o modelo")

        tempo_inip=(time.time())#initial predict time
        y_pred=rf.predict(x_testarea)#predict
        tempo_finp=(time.time())#final predict time
        temppred[i,j]=tempo_finp-tempo_inip
        print(f"{temppred[i,j]} segundos para predizer com o modelo")

        r2_scorerf[i,j]=r2_score(y_testarea,y_pred)#evaluate r2_score
        print('R2_Score='+str(r2_scorerf[i,j]))
        MAErf[i,j]=mean_absolute_error(y_testarea, y_pred)#evaluate MAE
        print('MAE='+str(MAErf[i,j]))
        RMSErf[i,j]=mean_squared_error(y_testarea, y_pred, squared=False)#evaluate RMSE
        print('RMSE='+str(RMSErf[i,j]))

        i=i+1

Repetition: 1
Metrics for RF: max_depth=5  max_samples=0.85  max_leaf_nodes=8000
73.43908548355103 segundos para treinar o modelo
0.21776938438415527 segundos para predizer com o modelo
R2_Score=0.9165033568172765
MAE=6.312580518959166
RMSE=11.147662801563202
Metrics for RF: max_depth=5  max_samples=0.85  max_leaf_nodes=400


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


70.0364818572998 segundos para treinar o modelo
0.18768763542175293 segundos para predizer com o modelo
R2_Score=0.9163817898594349
MAE=6.319963734210046
RMSE=11.15577507104841
Metrics for RF: max_depth=5  max_samples=0.85  max_leaf_nodes=1500


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


72.45780372619629 segundos para treinar o modelo
0.1695718765258789 segundos para predizer com o modelo
R2_Score=0.9161166300636565
MAE=6.3326029076231745
RMSE=11.173448982532058
Metrics for RF: max_depth=5  max_samples=0.85  max_leaf_nodes=3000


...

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


48.21120047569275 segundos para treinar o modelo
0.15747761726379395 segundos para predizer com o modelo
R2_Score=0.9173679704277613
MAE=6.271410655788671
RMSE=11.08979518990328
Metrics for RF: max_depth=5  max_samples=0.5  max_leaf_nodes=3000


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


46.98845911026001 segundos para treinar o modelo
0.17092275619506836 segundos para predizer com o modelo
R2_Score=0.917308545281998
MAE=6.282198794998652
RMSE=11.093782107922562


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [ ]:
print("R2 Score: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(r2_scorerf[i,j])

print("MAE: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(MAErf[i,j])

print("RMSErf: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
   print(RMSErf[i,j])


print("Tempo de Treinamento: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(temptrain[i,j])

print("Tempo de Predição: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(temppred[i,j])

R2 Score: 
repetition0
0.9165033568172765
0.9163817898594349
0.9161166300636565
0.9165412274426573
0.9167585521444919
0.9167615089947285
0.9166803497403561
0.916702663767226
0.9174810837462634
0.9168840364078458
0.917182405280653
0.9174432247128467
repetition1
0.9164279914633828
0.916674859895741
0.9166059746143427
0.9165105829744695
0.9165049697774619
0.9169925954693606
0.9168248630022576
0.9172062131543747
0.9169693077790498
0.9169951823447261
0.9170600534169918
0.9171144316917319
repetition2
0.9164958687101119
0.916664829680022
0.9165547000345114
0.9164918173545121
0.9170392879258595
0.9166664342030152
0.917002430751556
0.91669134942775
0.9170467441391217
0.9171721247605191
0.9169150604582786
0.9173294842270209
repetition3
0.9167916236147963
0.9163744726301797
0.9163833184993277
0.9163616446139267
0.916971452010412
0.9166969767508237
0.9170501136952895
0.9165514556485104
0.9169867669198257
0.9171165080872054
0.9171377119420401
0.9172726975748821
repetition4
0.9162400192051952
0.9163

### ANOVA COMB - melhores valores (grupo k)

In [ ]:
#Study Combination - max_depth+max_samples+max_leaf_nodes - piores resultados
import pickle#Save models
modelspath='/content/drive/MyDrive/Doutorado - UFBA/modelos_randomforest/'
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error


ndepth=[15,16]
maxsamples=[0.7, 0.85]
maxleafn=[3000,1500,8000]

reps=5#number of repetitions
shapetest=(len(ndepth)*len(maxsamples)*len(maxleafn),reps)
r2_scorerf=np.zeros(shapetest)#r2_score
MAErf=np.zeros(shapetest)#Mean Absolute Error
RMSErf=np.zeros(shapetest)# Root Mean Squared Error
oob_scorerf=np.zeros(shapetest)# OOB Score
temptrain=np.zeros(shapetest)# Tempo de treinamento
temppred=np.zeros(shapetest)# Tempo de predição


j=0

for j in range(reps):
  print('Repetition: '+str(j+1))
  i=0
  for ndep in ndepth:
    for maxsamp in maxsamples:
      for maxlf in maxleafn:
        rf = RandomForestRegressor(max_depth=ndep,max_samples=maxsamp,max_leaf_nodes=maxlf,n_estimators=800,
                                   max_features=12,min_samples_split=9,criterion='friedman_mse',
                                   min_samples_leaf=1)
        print('Metrics for RF: '+'max_depth='+str(ndep)+'  max_samples='+str(maxsamp)+'  max_leaf_nodes='+str(maxlf))
        tempo_init=(time.time())#initial training time
        rf.fit(x_trainarea, y_trainarea)#fit the RF model
        tempo_fint=(time.time())#final training time
        temptrain[i,j]=tempo_fint-tempo_init
        print(f"{temptrain[i,j]} segundos para treinar o modelo")

        tempo_inip=(time.time())#initial predict time
        y_pred=rf.predict(x_testarea)#predict
        tempo_finp=(time.time())#final predict time
        temppred[i,j]=tempo_finp-tempo_inip
        print(f"{temppred[i,j]} segundos para predizer com o modelo")

        r2_scorerf[i,j]=r2_score(y_testarea,y_pred)#evaluate r2_score
        print('R2_Score='+str(r2_scorerf[i,j]))
        MAErf[i,j]=mean_absolute_error(y_testarea, y_pred)#evaluate MAE
        print('MAE='+str(MAErf[i,j]))
        RMSErf[i,j]=mean_squared_error(y_testarea, y_pred, squared=False)#evaluate RMSE
        print('RMSE='+str(RMSErf[i,j]))

        i=i+1

Repetition: 1
Metrics for RF: max_depth=15  max_samples=0.7  max_leaf_nodes=3000
154.07664561271667 segundos para treinar o modelo
0.5996253490447998 segundos para predizer com o modelo
R2_Score=0.9315131007789182
MAE=5.4100980999329575
RMSE=10.096086608589856
Metrics for RF: max_depth=15  max_samples=0.7  max_leaf_nodes=1500


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


149.68464016914368 segundos para treinar o modelo
0.5621640682220459 segundos para predizer com o modelo
R2_Score=0.9313446004003032
MAE=5.409606321071891
RMSE=10.108498831552941
Metrics for RF: max_depth=15  max_samples=0.7  max_leaf_nodes=8000


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


150.37108325958252 segundos para treinar o modelo
0.5866281986236572 segundos para predizer com o modelo
R2_Score=0.9316236060711746
MAE=5.407818685912257
RMSE=10.087938178612024
Metrics for RF: max_depth=15  max_samples=0.85  max_leaf_nodes=3000


...

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


165.9146933555603 segundos para treinar o modelo
0.511671781539917 segundos para predizer com o modelo
R2_Score=0.9317035519174758
MAE=5.410536406367423
RMSE=10.082039032840822
Metrics for RF: max_depth=16  max_samples=0.85  max_leaf_nodes=8000


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


173.40071511268616 segundos para treinar o modelo
0.6226859092712402 segundos para predizer com o modelo
R2_Score=0.9317968500653849
MAE=5.414173673916087
RMSE=10.07515026292752


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [ ]:
print("R2 Score: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(r2_scorerf[i,j])

print("MAE: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(MAErf[i,j])

print("RMSErf: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
   print(RMSErf[i,j])


print("Tempo de Treinamento: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(temptrain[i,j])

print("Tempo de Predição: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(temppred[i,j])

R2 Score: 
repetition0
0.9315131007789182
0.9313446004003032
0.9316236060711746
0.9314729500879496
0.9320376164513434
0.9318384837362252
0.9316411156532466
0.9313222535193019
0.9315091766381131
0.9318451229416737
0.931976517598134
0.9317196787919866
repetition1
0.931235205238831
0.9313052535330989
0.9316454066636599
0.9316982064067856
0.9317214042065309
0.93160426033752
0.9317334552460518
0.9317942504096957
0.931339672689345
0.9321641220005121
0.9313570922403176
0.9316243141491455
repetition2
0.9315509215935174
0.9313616758487684
0.9311976253496844
0.9315820051708517
0.9319262402956556
0.9318214647331391
0.9315224262932419
0.9315448446342725
0.931615730596
0.9318724972586971
0.9316173278302423
0.9316667072039437
repetition3
0.9316448111286331
0.9317043122888423
0.9315809111420682
0.931647953874691
0.93179517388597
0.9317219751776825
0.9314543305698736
0.9314036447406356
0.9318197024398749
0.9316679340422643
0.9313912806309668
0.9316328141047244
repetition4
0.9318101902570596
0.93164174

### ANOVA FATOR

#### Pior Combinação

In [ ]:
#Study Combination - max_depth+max_samples+max_leaf_nodes - pior combinação
import pickle#Save models
modelspath='/content/drive/MyDrive/Doutorado - UFBA/modelos_randomforest/'
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error


ndepth=[5]
maxsamples=[0.25]
maxleafn=[400]

reps=5#number of repetitions
shapetest=(len(ndepth)*len(maxsamples)*len(maxleafn),reps)
r2_scorerf=np.zeros(shapetest)#r2_score
MAErf=np.zeros(shapetest)#Mean Absolute Error
RMSErf=np.zeros(shapetest)# Root Mean Squared Error
oob_scorerf=np.zeros(shapetest)# OOB Score
temptrain=np.zeros(shapetest)# Tempo de treinamento
temppred=np.zeros(shapetest)# Tempo de predição


j=0

for j in range(reps):
  print('Repetition: '+str(j+1))
  i=0
  for ndep in ndepth:
    for maxsamp in maxsamples:
      for maxlf in maxleafn:
        rf = RandomForestRegressor(max_depth=ndep,max_samples=maxsamp,max_leaf_nodes=maxlf,n_estimators=800,
                                   max_features=12,min_samples_split=9,criterion='friedman_mse',
                                   min_samples_leaf=1)
        print('Metrics for RF: '+'max_depth='+str(ndep)+'  max_samples='+str(maxsamp)+'  max_leaf_nodes='+str(maxlf))
        tempo_init=(time.time())#initial training time
        rf.fit(x_trainarea, y_trainarea)#fit the RF model
        tempo_fint=(time.time())#final training time
        temptrain[i,j]=tempo_fint-tempo_init
        print(f"{temptrain[i,j]} segundos para treinar o modelo")

        tempo_inip=(time.time())#initial predict time
        y_pred=rf.predict(x_testarea)#predict
        tempo_finp=(time.time())#final predict time
        temppred[i,j]=tempo_finp-tempo_inip
        print(f"{temppred[i,j]} segundos para predizer com o modelo")

        r2_scorerf[i,j]=r2_score(y_testarea,y_pred)#evaluate r2_score
        print('R2_Score='+str(r2_scorerf[i,j]))
        MAErf[i,j]=mean_absolute_error(y_testarea, y_pred)#evaluate MAE
        print('MAE='+str(MAErf[i,j]))
        RMSErf[i,j]=mean_squared_error(y_testarea, y_pred, squared=False)#evaluate RMSE
        print('RMSE='+str(RMSErf[i,j]))

        i=i+1

Repetition: 1
Metrics for RF: max_depth=5  max_samples=0.25  max_leaf_nodes=400
31.45571517944336 segundos para treinar o modelo
0.17132830619812012 segundos para predizer com o modelo
R2_Score=0.9175072700876113
MAE=6.269506763742141
RMSE=11.080443753744513
Repetition: 2
Metrics for RF: max_depth=5  max_samples=0.25  max_leaf_nodes=400


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


26.742491245269775 segundos para treinar o modelo
0.15253138542175293 segundos para predizer com o modelo
R2_Score=0.9176236323597888
MAE=6.263088400456852
RMSE=11.072626091422778
Repetition: 3
Metrics for RF: max_depth=5  max_samples=0.25  max_leaf_nodes=400


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


26.73503589630127 segundos para treinar o modelo
0.16435575485229492 segundos para predizer com o modelo
R2_Score=0.9177578210380357
MAE=6.262987279695269
RMSE=11.063603924716817
Repetition: 4
Metrics for RF: max_depth=5  max_samples=0.25  max_leaf_nodes=400


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


27.28551936149597 segundos para treinar o modelo
0.16163253784179688 segundos para predizer com o modelo
R2_Score=0.9174323493394078
MAE=6.286800279068552
RMSE=11.085474298460516
Repetition: 5
Metrics for RF: max_depth=5  max_samples=0.25  max_leaf_nodes=400


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


26.12516736984253 segundos para treinar o modelo
0.20639586448669434 segundos para predizer com o modelo
R2_Score=0.9176306792878631
MAE=6.257436482542591
RMSE=11.072152474591261


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [ ]:
print("R2 Score: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(r2_scorerf[i,j])

print("MAE: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(MAErf[i,j])

print("RMSErf: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
   print(RMSErf[i,j])


print("Tempo de Treinamento: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(temptrain[i,j])

print("Tempo de Predição: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(temppred[i,j])

R2 Score: 
repetition0
0.9175072700876113
repetition1
0.9176236323597888
repetition2
0.9177578210380357
repetition3
0.9174323493394078
repetition4
0.9176306792878631
MAE: 
repetition0
6.269506763742141
repetition1
6.263088400456852
repetition2
6.262987279695269
repetition3
6.286800279068552
repetition4
6.257436482542591
RMSErf: 
repetition0
11.080443753744513
repetition1
11.072626091422778
repetition2
11.063603924716817
repetition3
11.085474298460516
repetition4
11.072152474591261
Tempo de Treinamento: 
repetition0
31.45571517944336
repetition1
26.742491245269775
repetition2
26.73503589630127
repetition3
27.28551936149597
repetition4
26.12516736984253
Tempo de Predição: 
repetition0
0.17132830619812012
repetition1
0.15253138542175293
repetition2
0.16435575485229492
repetition3
0.16163253784179688
repetition4
0.20639586448669434


#### Melhor Combinação

In [ ]:
#Study Combination - max_depth+max_samples+max_leaf_nodes - melhor combinação - considerando um modelo menos complexo
import pickle#Save models
modelspath='/content/drive/MyDrive/Doutorado - UFBA/modelos_randomforest/'
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error


ndepth=[15]
maxsamples=[0.85]
maxleafn=[1500]

reps=5#number of repetitions
shapetest=(len(ndepth)*len(maxsamples)*len(maxleafn),reps)
r2_scorerf=np.zeros(shapetest)#r2_score
MAErf=np.zeros(shapetest)#Mean Absolute Error
RMSErf=np.zeros(shapetest)# Root Mean Squared Error
oob_scorerf=np.zeros(shapetest)# OOB Score
temptrain=np.zeros(shapetest)# Tempo de treinamento
temppred=np.zeros(shapetest)# Tempo de predição


j=0

for j in range(reps):
  print('Repetition: '+str(j+1))
  i=0
  for ndep in ndepth:
    for maxsamp in maxsamples:
      for maxlf in maxleafn:
        rf = RandomForestRegressor(max_depth=ndep,max_samples=maxsamp,max_leaf_nodes=maxlf,n_estimators=800,
                                   max_features=12,min_samples_split=9,criterion='friedman_mse',
                                   min_samples_leaf=1)
        print('Metrics for RF: '+'max_depth='+str(ndep)+'  max_samples='+str(maxsamp)+'  max_leaf_nodes='+str(maxlf))
        tempo_init=(time.time())#initial training time
        rf.fit(x_trainarea, y_trainarea)#fit the RF model
        tempo_fint=(time.time())#final training time
        temptrain[i,j]=tempo_fint-tempo_init
        print(f"{temptrain[i,j]} segundos para treinar o modelo")

        tempo_inip=(time.time())#initial predict time
        y_pred=rf.predict(x_testarea)#predict
        tempo_finp=(time.time())#final predict time
        temppred[i,j]=tempo_finp-tempo_inip
        print(f"{temppred[i,j]} segundos para predizer com o modelo")

        r2_scorerf[i,j]=r2_score(y_testarea,y_pred)#evaluate r2_score
        print('R2_Score='+str(r2_scorerf[i,j]))
        MAErf[i,j]=mean_absolute_error(y_testarea, y_pred)#evaluate MAE
        print('MAE='+str(MAErf[i,j]))
        RMSErf[i,j]=mean_squared_error(y_testarea, y_pred, squared=False)#evaluate RMSE
        print('RMSE='+str(RMSErf[i,j]))

        i=i+1

Repetition: 1
Metrics for RF: max_depth=15  max_samples=0.85  max_leaf_nodes=1500
165.6066586971283 segundos para treinar o modelo
0.5348706245422363 segundos para predizer com o modelo
R2_Score=0.9318348985713489
MAE=5.409137172041084
RMSE=10.07233955774557
Repetition: 2
Metrics for RF: max_depth=15  max_samples=0.85  max_leaf_nodes=1500


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


166.65798807144165 segundos para treinar o modelo
0.5336344242095947 segundos para predizer com o modelo
R2_Score=0.9313801591858367
MAE=5.4104642071727715
RMSE=10.105880738160048
Repetition: 3
Metrics for RF: max_depth=15  max_samples=0.85  max_leaf_nodes=1500


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


166.90925765037537 segundos para treinar o modelo
0.6355037689208984 segundos para predizer com o modelo
R2_Score=0.9313348250403901
MAE=5.419572319917957
RMSE=10.10921844499719
Repetition: 4
Metrics for RF: max_depth=15  max_samples=0.85  max_leaf_nodes=1500


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


165.181645154953 segundos para treinar o modelo
0.5141019821166992 segundos para predizer com o modelo
R2_Score=0.9317311173046356
MAE=5.405524666586454
RMSE=10.080004202587
Repetition: 5
Metrics for RF: max_depth=15  max_samples=0.85  max_leaf_nodes=1500


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


164.9657003879547 segundos para treinar o modelo
0.5294382572174072 segundos para predizer com o modelo
R2_Score=0.9316699534430233
MAE=5.406745211177149
RMSE=10.084518660183507


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [ ]:
print("R2 Score: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(r2_scorerf[i,j])

print("MAE: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(MAErf[i,j])

print("RMSErf: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
   print(RMSErf[i,j])


print("Tempo de Treinamento: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(temptrain[i,j])

print("Tempo de Predição: ")
for j in range(reps):
  print('repetition'+str(j))
  for i in range(len(ndepth)*len(maxsamples)*len(maxleafn)):
    print(temppred[i,j])

R2 Score: 
repetition0
0.9318348985713489
repetition1
0.9313801591858367
repetition2
0.9313348250403901
repetition3
0.9317311173046356
repetition4
0.9316699534430233
MAE: 
repetition0
5.409137172041084
repetition1
5.4104642071727715
repetition2
5.419572319917957
repetition3
5.405524666586454
repetition4
5.406745211177149
RMSErf: 
repetition0
10.07233955774557
repetition1
10.105880738160048
repetition2
10.10921844499719
repetition3
10.080004202587
repetition4
10.084518660183507
Tempo de Treinamento: 
repetition0
165.6066586971283
repetition1
166.65798807144165
repetition2
166.90925765037537
repetition3
165.181645154953
repetition4
164.9657003879547
Tempo de Predição: 
repetition0
0.5348706245422363
repetition1
0.5336344242095947
repetition2
0.6355037689208984
repetition3
0.5141019821166992
repetition4
0.5294382572174072
